# VCP Scanner — Volatility Contraction Pattern

Detects VCP setups as defined in https://tikamalma.substack.com/p/understanding-basics-of-vcp-and-creating

**Pattern criteria:**
1. Prior uptrend: price ≥ 30% above 52-week low, close above MA50
2. Base duration: ≥ 30 trading bars
3. Contractions: ≥ 2, each retracement ≤ 60% of the previous one
4. Keltner Channel width decreasing across contractions
5. Volume drying up during contractions (< 70% of 20-day avg)
6. ATR reduced ≥ 20% over the base
7. Breakout pivot: close near base high with volume spike ≥ 1.5×

In [13]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import talib

sys.path.insert(0, str(pathlib.Path('..').resolve()))
from backend.app.services.data_loader import load_stocks

plt.style.use('dark_background')
print('imports ok')

imports ok


In [14]:
from backend.app.services.data_loader import load_stocks

BENCHMARK = 'VNINDEX'

df_raw = load_stocks()

open_  = df_raw['open']
high   = df_raw['high']
low    = df_raw['low']
close  = df_raw['close']
volume = df_raw['volume']

valid_syms = close.columns[close.isna().mean() < 0.2]
close  = close[valid_syms].ffill()
high   = high[valid_syms].ffill()
low    = low[valid_syms].ffill()
open_  = open_[valid_syms].ffill()
volume = volume[valid_syms].ffill().fillna(0)

if BENCHMARK in close.columns:
    bm_close  = close[BENCHMARK]
    bm_volume = volume[BENCHMARK]
else:
    bm_close  = close.mean(axis=1)
    bm_volume = volume.mean(axis=1)
    print(f'BENCHMARK {BENCHMARK!r} not found — using equal-weight index')

print(f'Shape : {close.shape}  |  {close.index[0].date()} → {close.index[-1].date()}')
print(f'Symbols: {len(valid_syms)}  |  Benchmark: {BENCHMARK}')
close.tail(3)


Loading from HDF5 cache: stocks_data_latest.h5
Shape: (2501, 990)  |  2016-06-20 → 2026-06-19
Shape : (2501, 198)  |  2016-06-20 → 2026-06-19
Symbols: 198  |  Benchmark: VNINDEX


symbol,AAA,ABB,ACB,ACV,ADS,AGG,AGR,ANV,APH,ASM,...,VNINDEX,VNM,VOS,VPB,VPG,VPI,VRE,VSC,VTP,YEG
date,,,,,,,,,,,,,,,,,,,,,
2026-06-17,7.40,16.9,22.0,44.6,9.48,12.05,15.00,21.90,5.69,5.93,...,1806.20,59.0,12.85,26.5,2.93,60.5,28.15,20.25,65.5,8.88
2026-06-18,7.38,17.2,22.4,44.5,9.43,11.90,14.75,21.80,5.61,5.90,...,1830.47,59.2,12.65,26.4,2.90,60.0,30.10,19.90,65.5,8.73
2026-06-19,7.43,17.4,22.2,44.2,9.38,11.90,15.05,21.45,5.54,5.87,...,1824.53,59.0,12.65,25.9,2.89,58.9,29.35,19.20,65.0,8.64


## VCP Detection Functions

In [15]:
# ── Scanner parameters ────────────────────────────────────────────────────────
BASE_WINDOW        = 120   # bars to look back for the base
PIVOT_STRENGTH     = 5     # bars each side for local high/low detection
MIN_CONTRACTIONS   = 2     # minimum valid contractions
CONTRACTION_RATIO  = 0.40  # each retracement must be ≤ 60% of previous
MIN_BASE_BARS      = 20    # minimum base duration
MIN_UPTREND_PCT    = 0.30  # close must be ≥ 30% above 52-week low
VOL_DRYUP_RATIO    = 0.70  # volume during contraction < 70% of 20d avg
ATR_REDUCTION      = 0.20  # ATR must shrink ≥ 20% from base start to end
BREAKOUT_VOL_MULT  = 1.50  # breakout volume ≥ 1.5× 20-day average
KC_ATR_PERIOD      = 14
KC_WINDOW          = 20
KC_MULT            = 2.0
ADX_PERIOD         = 14
ADX_MIN            = 20    # minimum ADX for trend strength
NEAR_HIGH_PCT      = 0.05  # close within 5% of base high = near pivot
# ─────────────────────────────────────────────────────────────────────────────


def find_pivot_highs(high_arr: np.ndarray, order: int = 5) -> np.ndarray:
    """Returns boolean mask of local highs (highest within ±order bars)."""
    n = len(high_arr)
    mask = np.zeros(n, dtype=bool)
    for i in range(order, n - order):
        window = high_arr[i - order: i + order + 1]
        if high_arr[i] == window.max() and (window == high_arr[i]).sum() == 1:
            mask[i] = True
    return mask


def find_pivot_lows(low_arr: np.ndarray, order: int = 5) -> np.ndarray:
    """Returns boolean mask of local lows (lowest within ±order bars)."""
    n = len(low_arr)
    mask = np.zeros(n, dtype=bool)
    for i in range(order, n - order):
        window = low_arr[i - order: i + order + 1]
        if low_arr[i] == window.min() and (window == low_arr[i]).sum() == 1:
            mask[i] = True
    return mask


def extract_contractions(high_arr, low_arr, pivot_h_mask, pivot_l_mask):
    """
    Build alternating high→low swings from pivot masks.

    Returns list of dicts: {hi_idx, lo_idx, hi_price, lo_price, retracement}
    where retracement = (hi - lo) / hi.
    """
    hi_idxs = np.where(pivot_h_mask)[0]
    lo_idxs = np.where(pivot_l_mask)[0]

    contractions = []
    for hi in hi_idxs:
        # first pivot low that comes after this high
        lo_after = lo_idxs[lo_idxs > hi]
        if len(lo_after) == 0:
            continue
        lo = lo_after[0]
        hi_price = high_arr[hi]
        lo_price = low_arr[lo]
        if hi_price <= 0:
            continue
        retracement = (hi_price - lo_price) / hi_price
        contractions.append({
            'hi_idx':      hi,
            'lo_idx':      lo,
            'hi_price':    hi_price,
            'lo_price':    lo_price,
            'retracement': retracement,
        })

    # keep only non-overlapping (ascending hi_idx)
    seen, clean = set(), []
    for c in contractions:
        if c['hi_idx'] not in seen:
            seen.add(c['hi_idx'])
            clean.append(c)
    return clean


def scan_symbol(sym: str,
                close_s, high_s, low_s, volume_s,
                kc_width_s, atr_s, adx_s, vol_ma20_s) -> dict | None:
    """
    Scan a single symbol for VCP setup using the last BASE_WINDOW bars.
    Returns a result dict or None if pattern not found.
    """
    n = len(close_s)
    if n < BASE_WINDOW + 252:
        return None

    # slice to base window
    c   = close_s.values[-BASE_WINDOW:]
    h   = high_s.values[-BASE_WINDOW:]
    l   = low_s.values[-BASE_WINDOW:]
    v   = volume_s.values[-BASE_WINDOW:]
    kc  = kc_width_s.values[-BASE_WINDOW:]
    atr = atr_s.values[-BASE_WINDOW:]
    adx = adx_s.values[-BASE_WINDOW:]
    vm  = vol_ma20_s.values[-BASE_WINDOW:]

    if np.isnan(c).any() or np.isnan(h).any() or np.isnan(l).any():
        return None

    # ── 1. Prior uptrend ──────────────────────────────────────────────────────
    year_low = close_s.values[-252:].min()
    if year_low <= 0 or c[-1] < year_low * (1 + MIN_UPTREND_PCT):
        return None

    ma50 = talib.SMA(close_s.values, timeperiod=50)[-1]
    if np.isnan(ma50) or c[-1] < ma50:
        return None

    if not np.isnan(adx[-1]) and adx[-1] < ADX_MIN:
        return None

    # ── 2. Base duration ──────────────────────────────────────────────────────
    base_high = h.max()
    base_low  = l.min()
    # base starts where price first reached base_high
    base_start_idx = int(np.argmax(h == base_high))
    base_bars = BASE_WINDOW - base_start_idx
    if base_bars < MIN_BASE_BARS:
        return None

    base_slice = slice(base_start_idx, None)
    h_base = h[base_slice]
    l_base = l[base_slice]
    v_base = v[base_slice]
    vm_base = vm[base_slice]
    kc_base = kc[base_slice]
    atr_base = atr[base_slice]

    # ── 3. Find contractions ──────────────────────────────────────────────────
    ph = find_pivot_highs(h_base, order=PIVOT_STRENGTH)
    pl = find_pivot_lows(l_base,  order=PIVOT_STRENGTH)
    contractions = extract_contractions(h_base, l_base, ph, pl)

    if len(contractions) < MIN_CONTRACTIONS:
        return None

    # ── 4. Each retracement ≤ 60% of previous ────────────────────────────────
    valid_retr = all(
        contractions[i]['retracement'] <= contractions[i-1]['retracement'] * CONTRACTION_RATIO
        for i in range(1, len(contractions))
    )
    if not valid_retr:
        return None

    # ── 5. KC width decreasing across contractions ────────────────────────────
    for i, ct in enumerate(contractions):
        ct['kc_width'] = float(np.nanmean(kc_base[ct['hi_idx']:ct['lo_idx'] + 1]))

    valid_kc = all(
        contractions[i]['kc_width'] < contractions[i-1]['kc_width']
        for i in range(1, len(contractions))
    )
    if not valid_kc:
        return None

    # ── 6. Volume drying up in most recent contraction ────────────────────────
    last_ct = contractions[-1]
    v_during = v_base[last_ct['hi_idx']:last_ct['lo_idx'] + 1]
    vm_during = vm_base[last_ct['hi_idx']:last_ct['lo_idx'] + 1]
    valid_vm = np.nanmean(vm_during) > 0 and (
        np.nanmean(v_during) < np.nanmean(vm_during) * VOL_DRYUP_RATIO
    )
    if not valid_vm:
        return None

    # ── 7. ATR reduction ≥ 20% ────────────────────────────────────────────────
    atr_start = np.nanmean(atr_base[:10])
    atr_end   = np.nanmean(atr_base[-10:])
    if atr_start <= 0 or (atr_start - atr_end) / atr_start < ATR_REDUCTION:
        return None

    # ── 8. Near pivot (close within NEAR_HIGH_PCT of base high) ──────────────
    near_pivot = c[-1] >= base_high * (1 - NEAR_HIGH_PCT)

    # ── 9. Recent volume vs 20d avg ────────────────────────────────────────────
    recent_vol_ratio = v[-1] / vm[-1] if vm[-1] > 0 else 0.0

    return {
        'symbol':          sym,
        'last_close':      round(float(c[-1]), 2),
        'base_high':       round(float(base_high), 2),
        'base_low':        round(float(base_low), 2),
        'base_bars':       int(base_bars),
        'n_contractions':  len(contractions),
        'last_retracement':round(float(last_ct['retracement']) * 100, 1),
        'atr_reduction':   round((atr_start - atr_end) / atr_start * 100, 1),
        'kc_width_latest': round(float(kc_base[-1]), 4),
        'vol_ratio':       round(float(recent_vol_ratio), 2),
        'near_pivot':      near_pivot,
        'adx':             round(float(adx[-1]), 1) if not np.isnan(adx[-1]) else None,
        'contractions':    contractions,
    }


print('VCP functions defined')

VCP functions defined


## Run Scanner

In [16]:
# Pre-compute indicators for all symbols at once (vectorised)
print('Computing indicators…')

kc_width_all = {}
atr_all      = {}
adx_all      = {}
vol_ma20_all = {}

for sym in close.columns:
    c_np = close[sym].values.astype(np.float64)
    h_np = high[sym].values.astype(np.float64)
    l_np = low[sym].values.astype(np.float64)
    v_np = volume[sym].values.astype(np.float64)

    atr_np  = talib.ATR(h_np, l_np, c_np, timeperiod=KC_ATR_PERIOD)
    ema_np  = talib.EMA(c_np, timeperiod=KC_WINDOW)
    kc_w    = (atr_np * KC_MULT * 2) / ema_np   # normalised KC width

    kc_width_all[sym] = pd.Series(kc_w,   index=close.index)
    atr_all[sym]      = pd.Series(atr_np, index=close.index)
    adx_all[sym]      = pd.Series(talib.ADX(h_np, l_np, c_np, timeperiod=ADX_PERIOD), index=close.index)
    vol_ma20_all[sym] = pd.Series(talib.SMA(v_np, timeperiod=20), index=close.index)

print('Scanning symbols…')
results = []
for sym in close.columns:
    r = scan_symbol(
        sym,
        close[sym], high[sym], low[sym], volume[sym],
        kc_width_all[sym], atr_all[sym], adx_all[sym], vol_ma20_all[sym],
    )
    if r:
        results.append(r)

print(f'\nFound {len(results)} VCP candidates out of {len(close.columns)} symbols.')

Computing indicators…
Scanning symbols…

Found 0 VCP candidates out of 198 symbols.


## Results Table

In [17]:
if not results:
    print('No VCP candidates found. Try relaxing parameters.')
else:
    cols = ['symbol', 'last_close', 'base_high', 'base_low', 'base_bars',
            'n_contractions', 'last_retracement', 'atr_reduction',
            'vol_ratio', 'adx', 'near_pivot']
    df_results = pd.DataFrame(results)[cols].copy()
    df_results['last_retracement'] = df_results['last_retracement'].map('{:.1f}%'.format)
    df_results['atr_reduction']    = df_results['atr_reduction'].map('{:.1f}%'.format)
    df_results['vol_ratio']        = df_results['vol_ratio'].map('{:.2f}x'.format)
    df_results = df_results.sort_values(['near_pivot', 'n_contractions'], ascending=[False, False])
    df_results = df_results.reset_index(drop=True)

    print(f'=== VCP Candidates ({len(df_results)}) — sorted by near_pivot then contractions ===')
    from IPython.display import display
    display(df_results)

No VCP candidates found. Try relaxing parameters.


## Chart — Top Candidates

In [18]:
MAX_CHARTS = 9   # max symbols to plot

plot_syms = [r['symbol'] for r in results
             if r['near_pivot']][:MAX_CHARTS]
if not plot_syms:
    plot_syms = [r['symbol'] for r in results][:MAX_CHARTS]

ncols = 3
nrows = (len(plot_syms) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = np.array(axes).flatten()

for ax, sym in zip(axes, plot_syms):
    r   = next(x for x in results if x['symbol'] == sym)
    idx = close.index[-BASE_WINDOW:]
    c_  = close[sym].values[-BASE_WINDOW:]
    h_  = high[sym].values[-BASE_WINDOW:]
    l_  = low[sym].values[-BASE_WINDOW:]
    v_  = volume[sym].values[-BASE_WINDOW:]
    vm_ = vol_ma20_all[sym].values[-BASE_WINDOW:]

    # find base start
    base_start_i = int(np.argmax(h_ == r['base_high']))
    base_slice   = slice(base_start_i, None)

    ph_mask = find_pivot_highs(h_[base_slice], order=PIVOT_STRENGTH)
    pl_mask = find_pivot_lows(l_[base_slice],  order=PIVOT_STRENGTH)
    ph_abs  = np.where(ph_mask)[0] + base_start_i
    pl_abs  = np.where(pl_mask)[0] + base_start_i

    ax.set_facecolor('#0d1117')
    ax.plot(idx, c_, color='#58a6ff', lw=1.2, label='close')

    # contraction pivot markers
    ax.scatter(idx[ph_abs], h_[ph_abs], marker='v', color='#ff6b6b', s=60, zorder=5)
    ax.scatter(idx[pl_abs], l_[pl_abs], marker='^', color='#51cf66', s=60, zorder=5)

    # base high line
    ax.axhline(r['base_high'], color='#ffd43b', lw=0.8, ls='--', alpha=0.7, label=f'pivot {r["base_high"]}')

    # shade base region
    ax.axvspan(idx[base_start_i], idx[-1], alpha=0.07, color='#339af0')

    # volume subplot via twin axis
    ax2 = ax.twinx()
    colors = ['#51cf66' if c_[i] >= c_[i-1] else '#ff6b6b' for i in range(len(c_))]
    ax2.bar(idx, v_, color=colors, alpha=0.25, width=0.8)
    ax2.plot(idx, vm_, color='#ffa94d', lw=0.8, ls=':')
    ax2.set_ylim(0, v_.max() * 4)
    ax2.set_yticks([])

    pivot_tag = ' ★ NEAR PIVOT' if r['near_pivot'] else ''
    ax.set_title(f"{sym}{pivot_tag}  |  {r['n_contractions']}C  ATR↓{r['atr_reduction']}%",
                 fontsize=10, color='white')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b%y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.legend(fontsize=7, loc='upper left')

for ax in axes[len(plot_syms):]:
    ax.set_visible(False)

fig.suptitle('VCP Candidates', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1800x0 with 0 Axes>

## Single Symbol Deep Dive

In [ ]:
INSPECT = results[0]['symbol'] if results else None

if INSPECT:
    r = next(x for x in results if x['symbol'] == INSPECT)
    print(f'=== {INSPECT} ===')
    for k, v_ in r.items():
        if k != 'contractions':
            print(f'  {k:25s}: {v_}')
    print(f'\n  Contractions:')
    for i, ct in enumerate(r['contractions']):
        print(f'    [{i+1}] hi={ct["hi_price"]:.2f}  lo={ct["lo_price"]:.2f}  '
              f'retracement={ct["retracement"]*100:.1f}%  '
              f'kc_width={ct.get("kc_width", float("nan")):.4f}')